# Отчёт по модулю А
## Проектирование и разработка архитектуры инфраструктуры данных

| | |
|---|---|
| **Компетенция** | Аналитик транспортных данных |
| **Проект** | Платформа мониторинга опасных транспортных ситуаций |
| **Репозиторий** | `data_analitick_3_11_v3/module_A/` |
| **Критерии** | `05-Критерии_оценки _основа.xlsx` (макс. 15.5 баллов) |

> **Инструкция:** выполните `./scripts/run_module.sh`, сделайте скриншоты по разделам ниже и сохраните в папку `images/`.


## 1. Цель и контекст

Департамент транспорта города поручил построить единый аналитический контур: видеопотоки с камер → метрики транспортного потока, опасные ситуации, batch-аналитика и прогноз средней скорости.

В модуле А спроектирована и **физически реализована** инфраструктура: Medallion DWH (MinIO / PostgreSQL / ClickHouse), Apache Kafka, DDL, топики, consumer Kafka→Silver, архитектурная диаграмма, обоснование технологий.


## 2. Функциональные требования (критерий А.1)

| ID | Требование | Компонент |
|----|------------|-----------|
| F1 | Детекция ТС: тип, направление, трекинг | YOLOv8; `silver_detections`, `silver_track_history` |
| F2 | Потоковые метрики и инциденты | `streaming_metrics`, `gold_incidents` |
| F3 | Batch-агрегаты 30 мин | `gold_traffic_aggregates`; Airflow `traffic_batch_pipeline` |
| F4 | Прогноз скорости +30 мин | `gold_predictions`; Airflow `ml_speed_forecast` |
| F5 | Дашборд realtime/batch/forecast | Metabase → ClickHouse |
| F6 | Кадры в объектном хранилище | MinIO `bronze-frames`; `frame_path` |


## 3. Нефункциональные требования (критерий А.1)

| ID | Требование | Реализация | Файл:строки |
|----|------------|------------|-------------|
| NF1 | Нет потери данных при сбое БД | Kafka + rollback consumer | `kafka_to_silver.py:53-96` |
| NF2 | Масштабирование камер | `dim_cameras`, `cameras.yaml` | `dwh_silver_postgres.sql:64-68` |
| NF3 | Аномалии в логах, не в DWH | `logs/anomalies/` | модуль В |
| NF4 | Опоздавшие данные в batch | `gold_watermark` | `dwh_gold_clickhouse.sql:83-90` |
| NF5 | Масштабирование потока | Kafka partitions | `init_kafka_topics.sh:31-34` |


## 4. Архитектура и технологии (критерий А.2)

| Роль | Технология | Назначение |
|------|------------|------------|
| Bronze | **MinIO** | Кадры с bbox |
| Silver | **PostgreSQL** | Факты, справочники |
| Gold | **ClickHouse** | OLAP-витрины |
| Шина | **Apache Kafka** | `raw_detections` |
| Поток | **Python** consumers | Silver loader, streaming |
| Детекция | **YOLOv8** + ByteTrack | Модуль Б |
| Оркестрация | **Apache Airflow** | Batch, ML, DQ |
| BI | **Metabase** | Модуль Д |


## 5–6. Обоснование стека и подход к DWH (критерий А.2)

Полный текст: [`tech_stack.md`](tech_stack.md)

**Подход:** Medallion (слои) + Kimball (факты + `dim_*`).

**Аргументы:**
1. Кадры — в MinIO Bronze, не в PostgreSQL.
2. Kafka буферизует поток при сбое PG (`kafka_to_silver.py`).
3. ClickHouse Gold — OLAP для batch 30 мин и Metabase.

**Оркестрация Airflow:** batch ELT → `traffic_batch_pipeline`; ML → `ml_speed_forecast`.


## 7. Архитектурная диаграмма (критерий А.3)

Единая схема: `diagrams/00_platform_architecture.drawio`

**На диаграмме (чеклист А.3):**
- Слои: источники → обработка → Kafka → Bronze/Silver/Gold → потребители
- Аналитическое хранилище: ClickHouse + PostgreSQL
- Брокер: Kafka | Обработчики: YOLO, Stream, Airflow, Silver Loader
- Объектное хранилище: MinIO | Интеграция: RTSP, Metabase
- Подписанные потоки на стрелках (не на блоках)


---
### 📷 Скриншот 1

**Архитектурная диаграмма draw.io (00_platform_architecture)**

Сохраните файл как: `module_A/images/01_архитектурная.png`

![placeholder](images/01_placeholder.png)


## 8. Логическая модель DWH (критерий А.4)

| Сущность по критерию | Таблица | DDL | Строки |
|----------------------|---------|-----|--------|
| Справочник «Камеры» | `dim_cameras` | `dwh_silver_postgres.sql` | 7–15 |
| Справочник «Типы ТС» | `dim_vehicle_types` | тот же | 18–28 |
| «Трекеры» | `silver_track_history` | тот же | 52–62 |
| Факт детекции | `silver_detections` | тот же | 31–49 |
| Факт опасной ситуации | `gold_incidents` | `dwh_gold_clickhouse.sql` | 38–49 |
| Прогноз скорости | `gold_predictions` | тот же | 52–65 |
| Витрина realtime | `streaming_metrics` | тот же | 5–17 |
| Витрина batch 30m | `gold_traffic_aggregates` | тот же | 20–35 |

Подробно: [`sql/schema_mapping.md`](sql/schema_mapping.md)


---
### 📷 Скриншот 2

**PostgreSQL справочники dim_cameras dim_vehicle_types**

Сохраните файл как: `module_A/images/02_postgresql.png`

![placeholder](images/02_placeholder.png)


---
### 📷 Скриншот 3

**PostgreSQL silver_detections и silver_track_history**

Сохраните файл как: `module_A/images/03_postgresql.png`

![placeholder](images/03_placeholder.png)


---
### 📷 Скриншот 4

**ClickHouse SHOW TABLES FROM transport**

Сохраните файл как: `module_A/images/04_clickhouse.png`

![placeholder](images/04_placeholder.png)


## 9. Реализация инфраструктуры

| Компонент | Где в коде |
|-----------|------------|
| Kafka-топики | `scripts/init_kafka_topics.sh:31-34` |
| Kafka → PostgreSQL | `scripts/kafka_to_silver.py:19-31`, `53-96` |
| MinIO bucket | `bronze-frames` (модуль Б) |
| Инициализация | `scripts/run_module.sh` |


---
### 📷 Скриншот 5

**Kafka topics list raw_detections**

Сохраните файл как: `module_A/images/05_kafka.png`

![placeholder](images/05_placeholder.png)


---
### 📷 Скриншот 6

**kafka_to_silver consumer в работе или лог Silver**

Сохраните файл как: `module_A/images/06_kafka_to_silver.png`

![placeholder](images/06_placeholder.png)


## 10. Проверка на РМ

Выполните ячейку ниже или команды в терминале из папки `module_A`.


In [ ]:
# Опционально: проверка (раскомментируйте при наличии сервисов)
# !psql -U postgres -d silver -c "SELECT * FROM dim_vehicle_types;"
# !clickhouse-client --password user -q "SHOW TABLES FROM transport"
# !bash scripts/init_kafka_topics.sh
print("См. терминал: ./scripts/run_module.sh")


In [ ]:
%%bash
cd ~/data_analitick_3_11_v3/module_A
echo "=== PostgreSQL Silver ==="
PGPASSWORD="${PG_PASSWORD:-postgres}" psql -U "${PG_USER:-postgres}" -h "${PG_HOST:-localhost}" -d silver -c "\dt" 2>/dev/null || echo "(psql недоступен)"
echo "=== ClickHouse Gold ==="
clickhouse-client --password "${CH_PASSWORD:-user}" -q "SHOW TABLES FROM transport" 2>/dev/null || echo "(clickhouse недоступен)"
echo "=== Kafka topics ==="
bash scripts/init_kafka_topics.sh 2>/dev/null || echo "(kafka недоступен)"


---
### 📷 Скриншот 7

**Вывод psql и clickhouse-client после run_module**

Сохраните файл как: `module_A/images/07_вывод.png`

![placeholder](images/07_placeholder.png)


## 11. Выводы

В модуле А выполнено:
- FN/NFR в отчёте (разделы 2–3)
- Конкретные технологии и обоснование DWH (Kimball + Medallion)
- Единая архитектурная диаграмма с потоками
- Физический DWH: Silver/Gold DDL, Kafka, consumer, MinIO
- Модули Б–Д используют эту инфраструктуру

**Приложения:** `requirements.md`, `tech_stack.md`, `CRITERIA_DEMO.md`
